<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/Unsamble_enmbedding_RuColla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    roc_auc_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import re
import random
from tqdm import tqdm  # Добавлен прогресс-бар
import time  # Добавлен таймер


In [ ]:
# Конфигурация модели
MODEL_NAME = 'sberbank-ai/rugpt3small_based_on_gpt2'
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
CALIBRATION_MODEL = 'logistic_regression'


In [ ]:
# Инициализация моделей
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to('cuda' if torch.cuda.is_available() else 'cpu')
embedding_model = AutoModelForSequenceClassification.from_pretrained(EMBEDDING_MODEL).to('cuda' if torch.cuda.is_available() else 'cpu')
embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Функция для генерации семантических эмбеддингов
def get_embeddings(texts):
    inputs = embedding_tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    return outputs.logits.cpu().numpy()

In [ ]:
# Шаблоны промптов
prompt_templates = [
    """Определи, является ли предложение грамматически правильным. Ответь 0 или 1.
Примеры:
{examples}
Тест: "{sentence}" Ответ:""",

    """Лингвистический анализ. Оцени корректность конструкции:
{examples}
Анализ предложения "{sentence}": Результат -""",

    """[Задача] Классификация грамматической правильности (0-ошибка, 1-верно)
{examples}
Предложение: {sentence}
Класс:"""
]

In [ ]:
# Функция генерации промпта с семантическим отбором примеров
def generate_prompt(sentence, examples, template_num=0, num_shots=4):
    current_emb = get_embeddings([sentence])[0]
    examples_emb = get_embeddings(examples['sentence'].tolist())

    distances = np.dot(examples_emb, current_emb) / (
        np.linalg.norm(examples_emb, axis=1) * np.linalg.norm(current_emb)
    )

    indices = np.argsort(-distances)[:num_shots]
    selected_examples = examples.iloc[indices].to_dict('records')

    example_str = "\n".join([
        f'Предложение: "{ex["sentence"]}"\nОтвет: {ex["acceptable"]}'
        for ex in selected_examples
    ])
    return prompt_templates[template_num].format(
        examples=example_str,
        sentence=sentence
    )


In [ ]:
# Функция предсказания (без калибровки)
def predict(sentence, examples, template_num=0, num_shots=4):
    prompt = generate_prompt(sentence, examples, template_num, num_shots)
    inputs = tokenizer(prompt, return_tensors='pt', max_length=1024, truncation=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    match = re.search(r'\b[01]\b', answer[-20:])
    raw_pred = int(match.group(0)) if match else 0

    return raw_pred, 0.5  # Вероятность 0.5 по умолчанию

In [ ]:
# Ансамблирование моделей
class TemplateEnsemble:
    def __init__(self, templates, examples, num_shots=4):
        self.templates = templates
        self.examples = examples
        self.num_shots = num_shots

    def predict(self, sentence):
        predictions = []
        for template_num in range(len(prompt_templates)):
            pred, prob = predict(
                sentence,
                self.examples,
                template_num=template_num,
                num_shots=self.num_shots
            )
            predictions.append((pred, prob))
        return predictions

In [ ]:
# Основная функция экспериментов
def run_experiments():
    # Таймер
    start_time = time.time()

    train = pd.read_csv('in_domain_train.csv', usecols=['sentence', 'acceptable'])
    test = pd.read_csv('in_domain_dev.csv', usecols=['sentence', 'acceptable'])

    ensemble = TemplateEnsemble(templates=prompt_templates, examples=train, num_shots=4)

    all_predictions = []
    all_probs = []

    # Прогресс-бар с таймингом
    with tqdm(total=len(test), desc='Обработка тестовых данных') as pbar:
        for idx, text in enumerate(test['sentence']):
            template_preds = ensemble.predict(text)
            raw_preds = [p[0] for p in template_preds]
            probs = [p[1] for p in template_preds]

            final_pred = np.bincount(raw_preds).argmax()
            final_prob = np.mean(probs)

            all_predictions.append(final_pred)
            all_probs.append(final_prob)

            # Обновление прогресс-бара
            pbar.update(1)
            pbar.set_postfix({
                'Прогресс': f'{idx+1}/{len(test)}',
                'Время': f'{time.time() - start_time:.1f}с'
            })

    # Вывод результатов
    print("\nConfusion Matrix:")
    print(confusion_matrix(test['acceptable'], all_predictions))

    print("\nClassification Report:")
    print(classification_report(test['acceptable'], all_predictions))

    print(f"\nROC-AUC Score: {roc_auc_score(test['acceptable'], all_probs):.4f}")
    RocCurveDisplay.from_predictions(test['acceptable'], all_probs).plot()

    # Итоговое время
    elapsed_time = time.time() - start_time
    print(f"\nОбщее время выполнения: {elapsed_time:.1f} секунд")

In [ ]:
run_experiments()

Обработка тестовых данных:   0%|          | 1/983 [31:16<511:46:00, 1876.13s/it, Прогресс=1/983, Время=1581.6с]


KeyboardInterrupt: 